In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import statsmodels.api as sm

# Create output folders inside 'project 2 future interns'
os.makedirs("data", exist_ok=True)
os.makedirs("visuals", exist_ok=True)

# 1. LOAD & CLEAN DATA
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Clean TotalCharges (contains blank spaces for tenure == 0 customers)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# Encode binary target and create analytical segments
df["Churn_Flag"] = (df["Churn"] == "Yes").astype(int)
df["Tenure Cohort"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=["0–6 Mos (New)", "7–12 Mos", "13–24 Mos", "25–48 Mos", "49–72 Mos (Loyal)"]
)

# Save cleaned dataset
df.to_csv("data/telco_customer_churn_cleaned.csv", index=False)

# 2. PRINT EXECUTIVE RETENTION & CHURN KPIs
total_customers = len(df)
churned_customers = df["Churn_Flag"].sum()
retained_customers = total_customers - churned_customers
churn_rate = (churned_customers / total_customers) * 100
retention_rate = 100 - churn_rate

monthly_rev_total = df["MonthlyCharges"].sum()
monthly_rev_lost = df.loc[df["Churn_Flag"] == 1, "MonthlyCharges"].sum()
annualized_rev_at_risk = monthly_rev_lost * 12
avg_tenure_retained = df.loc[df["Churn_Flag"] == 0, "tenure"].mean()
avg_tenure_churned = df.loc[df["Churn_Flag"] == 1, "tenure"].mean()
avg_clv = df["TotalCharges"].mean()

print("=" * 60)
print("EXECUTIVE RETENTION & CHURN KPI SUMMARY")
print("=" * 60)
print(f"Total Customers:             {total_customers:,}")
print(f"Retained Customers:          {retained_customers:,} ({retention_rate:.2f}%)")
print(f"Churned Customers:           {churned_customers:,} ({churn_rate:.2f}%)")
print(f"Monthly Recurring Revenue:   ${monthly_rev_total:,.2f}")
print(f"Monthly Revenue Lost (Churn):${monthly_rev_lost:,.2f}")
print(f"Annualized Revenue at Risk:  ${annualized_rev_at_risk:,.2f}")
print(f"Avg Tenure (Retained):       {avg_tenure_retained:.1f} months")
print(f"Avg Tenure (Churned):        {avg_tenure_churned:.1f} months")
print(f"Avg Customer Lifetime Value: ${avg_clv:,.2f}")
print("=" * 60)

print("\n--- CHURN BY CONTRACT TYPE ---")
contract_summary = df.groupby("Contract").agg(
    Customers=("customerID", "count"),
    Churned=("Churn_Flag", "sum"),
    Churn_Rate_Pct=("Churn_Flag", lambda x: x.mean() * 100),
    Monthly_Rev_Lost=("MonthlyCharges", lambda x: x[df.loc[x.index, "Churn_Flag"] == 1].sum())
)
print(contract_summary.round(2))

print("\n--- CHURN BY TENURE COHORT ---")
cohort_summary = df.groupby("Tenure Cohort", observed=False).agg(
    Customers=("customerID", "count"),
    Churn_Rate_Pct=("Churn_Flag", lambda x: x.mean() * 100),
    Avg_Monthly_Charge=("MonthlyCharges", "mean")
)
print(cohort_summary.round(2))

print("\n--- CHURN BY INTERNET SERVICE & PAYMENT METHOD ---")
print(df.groupby("InternetService")["Churn_Flag"].agg(Customers="count", Churn_Rate=lambda x: round(x.mean()*100, 2)))
print()
print(df.groupby("PaymentMethod")["Churn_Flag"].agg(Customers="count", Churn_Rate=lambda x: round(x.mean()*100, 2)))

# 3. STATISTICAL ANALYSIS: LOGISTIC REGRESSION (KEY CHURN DRIVERS)
model_df = pd.DataFrame({
    "tenure": df["tenure"],
    "MonthlyCharges": df["MonthlyCharges"],
    "Contract_1Yr": (df["Contract"] == "One year").astype(int),
    "Contract_2Yr": (df["Contract"] == "Two year").astype(int),
    "FiberOptic": (df["InternetService"] == "Fiber optic").astype(int),
    "ElectronicCheck": (df["PaymentMethod"] == "Electronic check").astype(int),
    "TechSupport_Yes": (df["TechSupport"] == "Yes").astype(int)
})
X = sm.add_constant(model_df)
y = df["Churn_Flag"]
logit_model = sm.Logit(y, X).fit(disp=False)

odds_ratios = pd.DataFrame({
    "Coef": logit_model.params,
    "Odds_Ratio": np.exp(logit_model.params),
    "P_Value": logit_model.pvalues
})
print("\n--- LOGISTIC REGRESSION: CHURN DRIVERS (ODDS RATIOS) ---")
print(odds_ratios.round(4))

# 4. GENERATE CLIENT-READY VISUALIZATIONS
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# CHART 1: Overall Churn Split & Monthly Revenue Impact
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
labels = [f"Retained ({retention_rate:.1f}%)", f"Churned ({churn_rate:.1f}%)"]
sizes = [retained_customers, churned_customers]
colors = ["#1f4e78", "#c0392b"]
axes[0].pie(sizes, labels=labels, colors=colors, startangle=90, wedgeprops=dict(width=0.45, edgecolor="white"), textprops=dict(fontweight="bold"))
axes[0].set_title("Customer Retention vs. Churn Split", fontsize=12, fontweight="bold")

rev_labels = ["Retained MRR", "Lost MRR (Churned)"]
rev_vals = [monthly_rev_total - monthly_rev_lost, monthly_rev_lost]
bars = axes[1].bar(rev_labels, rev_vals, color=colors, width=0.5)
axes[1].yaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
axes[1].set_title("Monthly Recurring Revenue (MRR) Impact", fontsize=12, fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
for bar in bars:
    h = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2, h + 6000, f"${h:,.0f}", ha="center", fontweight="bold")
axes[1].set_ylim(0, max(rev_vals) * 1.15)
plt.tight_layout()
plt.savefig("visuals/01_churn_overview_mrr_impact.png", dpi=300)
plt.close()

# CHART 2: Churn Rate by Contract Type & Payment Method
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

c_plot = contract_summary.reset_index().sort_values("Churn_Rate_Pct", ascending=False)
b1 = axes[0].bar(c_plot["Contract"], c_plot["Churn_Rate_Pct"], color=["#c0392b", "#e68a00", "#2e8b57"], width=0.52)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[0].set_title("Churn Rate (%) by Contract Type", fontsize=12, fontweight="bold")
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
for bar in b1:
    h = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2, h + 1.2, f"{h:.1f}%", ha="center", fontweight="bold")
axes[0].set_ylim(0, 52)

pm = df.groupby("PaymentMethod")["Churn_Flag"].mean().mul(100).sort_values(ascending=True)
short_pm = [p.replace(" (automatic)", "\n(auto)") for p in pm.index]
b2 = axes[1].barh(short_pm, pm.values, color=["#2e8b57", "#4a90d9", "#4a90d9", "#c0392b"], height=0.55)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].set_title("Churn Rate (%) by Payment Method", fontsize=12, fontweight="bold")
axes[1].grid(axis="x", linestyle="--", alpha=0.4)
for bar in b2:
    w = bar.get_width()
    axes[1].text(w + 1.0, bar.get_y() + bar.get_height()/2, f"{w:.1f}%", va="center", fontweight="bold")
axes[1].set_xlim(0, 55)

plt.tight_layout()
plt.savefig("visuals/02_churn_by_contract_and_payment.png", dpi=300)
plt.close()

# CHART 3: Customer Tenure Cohort Retention Curve
cohort_plot = cohort_summary.reset_index()
fig, ax = plt.subplots(figsize=(10, 4.8))
bars = ax.bar(cohort_plot["Tenure Cohort"], cohort_plot["Churn_Rate_Pct"], color="#2b5c8f", width=0.55)
ax.plot(cohort_plot["Tenure Cohort"], cohort_plot["Churn_Rate_Pct"], color="#c0392b", marker="o", linewidth=2)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Churn Rate (%) Across Customer Tenure Cohorts", fontsize=13, fontweight="bold", pad=12)
ax.set_ylabel("Churn Rate (%)")
ax.grid(axis="y", linestyle="--", alpha=0.4)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 1.5, f"{h:.1f}%", ha="center", fontweight="bold")
ax.set_ylim(0, 62)
plt.tight_layout()
plt.savefig("visuals/03_tenure_cohort_churn_curve.png", dpi=300)
plt.close()

# CHART 4: Monthly Charges Distribution (Retained vs. Churned)
fig, ax = plt.subplots(figsize=(9.5, 4.8))
sns.kdeplot(data=df[df["Churn"] == "No"], x="MonthlyCharges", fill=True, color="#1f4e78", alpha=0.4, label="Retained Customers", ax=ax)
sns.kdeplot(data=df[df["Churn"] == "Yes"], x="MonthlyCharges", fill=True, color="#c0392b", alpha=0.4, label="Churned Customers", ax=ax)
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter("${x:,.0f}"))
ax.set_title("Monthly Charges Distribution: Retained vs. Churned Customers", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Monthly Charges (USD)")
ax.set_ylabel("Customer Density")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("visuals/04_monthly_charges_distribution.png", dpi=300)
plt.close()

print("\nAll 4 Task 2 charts saved to visuals/ and cleaned CSV saved to data/!")

EXECUTIVE RETENTION & CHURN KPI SUMMARY
Total Customers:             7,043
Retained Customers:          5,174 (73.46%)
Churned Customers:           1,869 (26.54%)
Monthly Recurring Revenue:   $456,116.60
Monthly Revenue Lost (Churn):$139,130.85
Annualized Revenue at Risk:  $1,669,570.20
Avg Tenure (Retained):       37.6 months
Avg Tenure (Churned):        18.0 months
Avg Customer Lifetime Value: $2,279.73

--- CHURN BY CONTRACT TYPE ---
                Customers  Churned  Churn_Rate_Pct  Monthly_Rev_Lost
Contract                                                            
Month-to-month       3875     1655           42.71         120847.10
One year             1473      166           11.27          14118.45
Two year             1695       48            2.83           4165.30

--- CHURN BY TENURE COHORT ---
                   Customers  Churn_Rate_Pct  Avg_Monthly_Charge
Tenure Cohort                                                   
0–6 Mos (New)           1481           52.94        

In [ ]:
6+